# S.T.I.T.C.H — Floorplan Segmentation Training
Run each cell top to bottom.

try using google collab rather than using your machine locally to train the model

In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2 — Check GPU
!nvidia-smi

In [ ]:
# CELL 3 — Unzip dataset from Drive
import os

ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
EXTRACT_PATH = '/content/'

print('Unzipping dataset...')
!unzip -q {ZIP_PATH} -d {EXTRACT_PATH}
print('Done!')
print('Total images:', len(os.listdir('/content/dataset/images')))
print('Total masks: ', len(os.listdir('/content/dataset/masks')))

In [ ]:
# CELL 4 — Install dependencies (added albumentations for augmentation)
!pip install tqdm opencv-python-headless albumentations -q

In [ ]:
# CELL 5 — Model definition (unchanged)
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DoubleConv(3, 64)
        self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128)
        self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256)
        self.p3 = nn.MaxPool2d(2)
        self.b  = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b  = self.b(self.p3(d3))
        u3 = self.c3(torch.cat([self.u3(b), d3], dim=1))
        u2 = self.c2(torch.cat([self.u2(u3), d2], dim=1))
        u1 = self.c1(torch.cat([self.u1(u2), d1], dim=1))
        return self.out(u1)

print(' Model defined')

In [ ]:
# CELL 6 — Dataset WITH augmentation + stroke masks + hard negatives
import os
import cv2
import numpy as np
import random
import albumentations as A
import torch
from torch.utils.data import Dataset, DataLoader


train_transform = A.Compose([
    A.Rotate(limit=180, p=0.8),
    A.ElasticTransform(alpha=120, sigma=6, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
], additional_targets={'mask': 'mask'})


def add_hard_negatives(img, mask):
    """Stamp synthetic furniture shapes onto image, mark them as background."""
    h, w = mask.shape
    for _ in range(random.randint(2, 6)):
        cx = random.randint(20, w - 20)
        cy = random.randint(20, h - 20)
        sz = random.randint(8, 30)
        temp = np.zeros((h, w), np.uint8)
        if random.random() < 0.5:
            cv2.rectangle(temp, (cx - sz, cy - sz), (cx + sz, cy + sz), 1, -1)
        else:
            cv2.circle(temp, (cx, cy), sz, 1, -1)
        # Only zero out pixels that aren't already real walls
        mask[temp == 1] = 0
    return img, mask


class FloorplanDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_names = sorted(os.listdir(img_dir))
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.augment   = augment

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        name = self.img_names[idx]

        img = cv2.imread(os.path.join(self.img_dir, name))
        if img is None:
            return self.__getitem__(0)
        img = cv2.resize(img, (256, 256))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(self.mask_dir, name), cv2.IMREAD_UNCHANGED)
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.uint8)

        # Hard negatives — 50% of samples
        if self.augment and random.random() < 0.5:
            img, mask = add_hard_negatives(img, mask)

        # Albumentations augmentation
        if self.augment:
            aug   = train_transform(image=img, mask=mask)
            img   = aug['image']
            mask  = aug['mask']

        img  = img.astype(np.float32) / 255.0
        img  = torch.from_numpy(img).permute(2, 0, 1)
        mask = mask.astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask

print(' Dataset defined (augmentation ON, hard negatives ON)')

In [ ]:
# CELL 7 — Training
import torch.optim as optim
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

dataset = FloorplanDataset('/content/dataset/images', '/content/dataset/masks', augment=True)
print(f'Total images: {len(dataset)}')

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Total batches per epoch: {len(loader)}')

model     = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# pos_weight=3.0 — walls are sparse pixels; penalise missing them 3x more than false positives
# this is the main fix for single-line walls being ignored during training
loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([3.0]).to(device)
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

epochs    = 15
best_loss = float('inf')

print('\nStarting training...')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}')

    for imgs, masks in loop:
        imgs  = imgs.to(device)
        masks = masks.to(device)
        preds = model(imgs)
        loss  = loss_fn(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}')

    scheduler.step(avg_loss)

    torch.save(model.state_dict(), '/content/unet.pth')

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/unet_best.pth')
        print(f'   Best model updated (loss: {best_loss:.4f})')

print(f'\n TRAINING COMPLETE — best loss: {best_loss:.4f}')

In [ ]:
# CELL 8 — Save both weights to Google Drive
import shutil
shutil.copy('/content/unet.pth',      '/content/drive/MyDrive/unet.pth')
shutil.copy('/content/unet_best.pth', '/content/drive/MyDrive/unet_best.pth')
print('   Both weights saved to Google Drive!')
print('   unet.pth      — final epoch')
print('   unet_best.pth — best loss epoch')